# HackerNews Analytics Pro — v7 Predictive Research

This notebook trains a leakage-conscious high-score classifier and detects unusual engagement patterns. The demo dataset is synthetic; replace it with real collected data for research.

**Important:** this is retrospective classification, not guaranteed future virality prediction.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'hn_analytics.py').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from hn_data_loader import load_stories
from hn_predictive import train_virality_model, detect_engagement_anomalies

data = load_stories(ROOT / 'demo_stories.jsonl')
data.head()

## 1. Dataset profile

The score is used only to define the outcome class. Predictor features are title text, domain, story type, author, posting time and title length.

In [ ]:
display(data[['title', 'score', 'descendants', 'by', 'type']].describe(include='all'))
print(f'{len(data):,} stories')

## 2. Train chronological classifier

The high-score cutoff is calculated from the training partition only. The latest 20% is held out in chronological order.

In [ ]:
model, metrics, predictions = train_virality_model(data)
metrics

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(predictions.actual_high_score, predictions.predicted_high_score)
plt.title('Chronological holdout confusion matrix')
plt.show()

## 3. Inspect predicted probabilities

Probabilities are model scores, not calibrated guarantees. Performance can vary substantially with dataset size, time period and feed composition.

In [ ]:
display(predictions.sort_values('high_score_probability', ascending=False).head(15))

## 4. Detect unusual engagement

Isolation Forest flags observations that differ from the dataset's score, comment and title-length patterns. An anomaly is not necessarily a successful story or bad data.

In [ ]:
anomalies = detect_engagement_anomalies(data, contamination=0.05)
display(anomalies[anomalies.is_anomaly].head(20))

In [ ]:
ax = anomalies.sort_values('anomaly_score').plot(x='title', y='anomaly_score', kind='barh', figsize=(10, 6), legend=False)
ax.set_title('Engagement anomaly score')
ax.set_xlabel('Higher = more unusual')
plt.tight_layout()
plt.show()

## 5. Export results

Run `hn-analytics research --input demo_stories.jsonl --output-dir reports` from the repository root to export metrics, holdout predictions and anomaly flags.